# Milestone 7 - Industry Architecture Reflection

**Architecture:** Edge Deployment / TinyML.

I chose Edge Deployment/TinyML because it connects directly to the Milestone 4 TensorFlow Lite export and gives the capstone a practical deployment direction.

## 1: Architecture Summary

Edge Deployment/TinyML is the practice of running neural network inference close to where data is produced, such as on phones, embedded Linux devices, microcontrollers, cameras, sensors, or other low-power hardware. Instead of sending every input to a cloud server, the model runs locally. This architecture is useful when latency, privacy, cost, connectivity, or energy use matter.

The core structure is not a single neural network layer type like a CNN or RNN. It is a system architecture that combines a compact model, a constrained runtime, hardware-aware preprocessing, and a deployment format such as TensorFlow Lite or TensorFlow Lite Micro. Models used in edge systems are often designed with efficient components, including depthwise separable convolutions, small input resolutions, quantized weights, pruning, and mobile-friendly backbones such as MobileNet. MobileNets are especially relevant because they were designed around depthwise separable convolutions to reduce model size and latency for mobile and embedded vision applications.

Compared with the models used earlier, edge deployment adds a systems constraint: the model must be accurate enough while fitting real memory, latency, battery, and compatibility limits. The best model is not always the highest-accuracy model; it is the model that performs reliably within the deployment environment.

## 2: Industry Use Case

A practical industry use case is **on-device image classification or visual inspection** in settings where network access is limited or cloud inference is too slow or expensive. Examples include agriculture cameras identifying plant conditions in the field, warehouse scanners detecting damaged packages, mobile apps classifying images without uploading them, and edge cameras filtering events before sending only important alerts.

For this project, the closest use case is mobile or edge scene classification. A camera-equipped device could classify whether it is seeing a street, mountain, sea, forest, glacier, or building scene locally. Local inference avoids sending raw images to a server, which can reduce privacy risk and latency. TensorFlow Lite is a good match because it is designed for on-device machine learning on mobile, embedded, and edge devices. For even smaller microcontroller-class systems, TensorFlow Lite Micro removes many desktop/server assumptions and targets highly constrained hardware.

This approach is effective when the input is sensor data generated at the edge and when immediate decisions matter more than very large model capacity. In the Milestone 4 workflow, converting the Keras model to `.tflite` is the first step toward this kind of deployment.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACT_DIR = ROOT / "artifacts"
metrics_path = ARTIFACT_DIR / "milestone4_metrics.json"
tflite_path = ARTIFACT_DIR / "milestone4_scene_cnn.tflite"

summary = {
    "metrics_file_found": metrics_path.exists(),
    "tflite_file_found": tflite_path.exists(),
    "tflite_size_mb": round(tflite_path.stat().st_size / (1024 * 1024), 3) if tflite_path.exists() else None,
}

if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    summary["test_accuracy"] = metrics.get("test_accuracy")
    report = metrics.get("classification_report", {})
    rows = []
    for label, values in report.items():
        if isinstance(values, dict) and {"precision", "recall", "f1-score"}.issubset(values):
            rows.append({"class_or_average": label, **values})
    display(pd.DataFrame(rows))
else:
    print("Run milestone4_deep_learning_image_classification.ipynb first to generate metrics and TFLite artifacts.")

summary

,class_or_average,precision,recall,f1-score,support
0,buildings,0.655738,0.823799,0.730223,437.0
1,forest,0.982143,0.812236,0.889145,474.0
2,glacier,0.581707,0.862568,0.694829,553.0
3,mountain,0.759009,0.641905,0.695562,525.0
4,sea,0.927039,0.423529,0.581427,510.0
5,street,0.708185,0.794411,0.748824,501.0
6,macro avg,0.768970,0.726408,0.723335,3000.0
7,weighted avg,0.766616,0.724333,0.720554,3000.0


{'metrics_file_found': True,
 'tflite_file_found': True,
 'tflite_size_mb': 0.291,
 'test_accuracy': 0.7243333458900452}

## 3: Evaluation and Reflection

Edge deployment is a strong fit for the scene-classification model because the input is visual data that could be captured directly on a phone, drone, embedded camera, or field device. Running inference locally would reduce latency and avoid uploading raw images, which is important when images may contain people, buildings, addresses, vehicles, or location clues. It also supports use in low-connectivity environments.

The main limitation is that edge devices force trade-offs. A larger model may improve accuracy but exceed memory limits or drain battery. Quantization can reduce model size and improve speed, but it can also reduce accuracy for classes that already overlap visually, such as glacier versus mountain or street versus buildings. Edge models can also become brittle under domain shift: a device deployed in a new region, season, or lighting condition may see images that differ from the training set.

For scaling, I would define deployment thresholds before release: maximum model size, target latency, minimum per-class recall, and maximum acceptable confidence failure rate. I would also benchmark the `.tflite` model on the actual target hardware instead of assuming desktop performance transfers to mobile or embedded devices.

## 4: Connection to Capstone

This architecture could elevate the capstone by turning the project from a notebook-only classifier into a deployable AI system. A capstone built around scene or environmental image classification could compare three versions: a scratch CNN, a transfer-learning model, and a TFLite-optimized model. The final design would not be chosen only by validation accuracy. It would also consider model size, inference latency, privacy, energy use, and failure cases.

This connects to the capstone in several ways. The hypothesis can specify that a transfer-learning model will improve accuracy while a TFLite model will improve deployment feasibility. The empirical strategy can include preprocessing, model selection, fine-tuning, export, latency measurement, and class-level evaluation. The risk reflection can map concrete harms to model behavior, such as misclassification under poor lighting or overconfidence on unfamiliar locations. Visualizations should include at least a training curve, confusion matrix, example predictions, and failure-case examples.

## Reflection

The most important consequence of edge deployment is the tension between privacy and accountability. Local inference can protect privacy because raw images do not need to leave the device. However, it can also make monitoring harder: if every device runs independently, failures may be less visible to developers unless careful logging, consent, and update mechanisms are designed. In a real system, I would log only privacy-preserving metadata such as confidence distributions, model version, latency, and user-approved error reports. I would avoid storing raw images by default, provide a clear opt-in path for data sharing, and build a process for updating the model when field performance degrades.